In [1]:
from mlpui.model_loader import load_torch_file

In [2]:
sd,metadata = load_torch_file("../models/mlp/uma-s-1p1.pt",return_metadata=True)

In [3]:
def unet_prefix_from_state_dict(state_dict):
    candidates = ["model.diffusion_model.",
                  "model.model.",
                  "net.",
                  "module.backbone.",
                  ]
    counts = {k: 0 for k in candidates}
    for k in state_dict:
        for c in candidates:
            if k.startswith(c):
                counts[c] += 1
                break

    top = max(counts, key=counts.get)
    if counts[top] > 5:
        return top
    else:
        return "model." #aura flow and others


In [4]:
prefix = unet_prefix_from_state_dict(sd)
prefix

'module.backbone.'

In [5]:
def calculate_parameters(sd, prefix=""):
    params = 0
    for k in sd.keys():
        if k.startswith(prefix):
            w = sd[k]
            params += w.nelement()
    return params

In [6]:
calculate_parameters(sd,prefix=prefix)

146533699

In [7]:
def weight_dtype(sd, prefix=""):
    dtypes = {}
    for k in sd.keys():
        if k.startswith(prefix):
            w = sd[k]
            dtypes[w.dtype] = dtypes.get(w.dtype, 0) + w.numel()

    if len(dtypes) == 0:
        return None

    return max(dtypes, key=dtypes.get)
weight_dtype(sd,prefix=prefix)

torch.float32

In [8]:
from mlpui.model_detection import model_config_from_unet,count_blocks
mlp_config = model_config_from_unet(sd,prefix,use_base_if_no_match=True)
print(mlp_config)

NameError: name 'UMA' is not defined

In [9]:
mlp_config.supported_inference_dtypes

NameError: name 'mlp_config' is not defined

In [10]:
from mlpui.model_loader import load_torch_file
from mlpui.utils import calculate_parameters,_weight_dtype
import mlpui.model_management as model_management
from mlpui.model_detection import unet_prefix_from_state_dict,model_config_from_unet
import logging

mlp_model_prefix = unet_prefix_from_state_dict(sd)
parameters = calculate_parameters(sd, mlp_model_prefix)
weight_dtype = _weight_dtype(sd, mlp_model_prefix)
load_device = model_management.get_torch_device()

model_config = model_config_from_unet(sd, mlp_model_prefix,use_base_if_no_match=True, metadata=metadata)
if model_config is None:
    logging.warning("Warning, This is not a checkpoint file")

unet_weight_dtype = list(model_config.supported_inference_dtypes)
if model_config.quant_config is not None:
    weight_dtype = None


unet_dtype = model_management.unet_dtype(model_params=parameters, supported_dtypes=unet_weight_dtype, weight_dtype=weight_dtype)
manual_cast_dtype = model_management.unet_manual_cast(unet_dtype, load_device, model_config.supported_inference_dtypes)
model_config.set_inference_dtype(unet_dtype, manual_cast_dtype)

inital_load_device = model_management.unet_inital_load_device(parameters, unet_dtype)
model = model_config.get_model(sd, mlp_model_prefix, device=inital_load_device)

ERROR:root:no match {'model_name': 'uma', 'num_blocks': 4, 'has_mole': True, 'datasets': ['oc20', 'omol', 'omat', 'odac', 'omc']}


NotImplementedError: 